# Sparse-scaling benchmark: Fe

Bulk bcc Fe (example04), ferromagnetic along z with spin-orbit coupling, `nawf = 26` per
primitive cell (the ladder runs to d = 4, `nawf = 416`, for now). The magnetization breaks time reversal, so the
anomalous Hall conductivity σ_xy is non-zero; it is tested by the Berry-curvature route
(`anomalous_Hall`, with the magnetic circular dichroism) and by the Kubo route
(`linear_response(response='ahc')`). Fe also runs the Si properties on a magnetic metal, on the
bottom-of-the-spectrum window and on an interior window through E_F.

A cell doubled `d` times is still the same crystal, so every property per primitive cell must
look the same at every size. Each figure shows one sparse pipeline. The colored lines are its
sizes, and the dashed black line is the dense reference on the primitive cell (`reference_d00`).
The curves come from [observables.py](../../observables.py), which normalizes them per primitive
cell, and the figures from [plots.py](../../plots.py). The settings are in
[system.py](system.py). A pipeline that does not run a property has no panel for it.

**Before the first ladder** there are no measured notes here yet. The expectations below come
from a smoke test at a 12³ mesh, d ≤ 1 (2026-10-26):

- With the exact H(R), sparse d = 0 matches the dense reference exactly in the Boltzmann σ and S, to 1.8×10⁻³ in the anomalous Hall conductivity (both routes), 10⁻² in the magnetic circular dichroism and 10⁻¹³ in the Kubo σ. The band-curvature family differs at d = 0 already: Hall 1.5–10%, Nernst 2%, effective masses 0.66.
- d = 1 matches d = 0 to 10⁻⁶ or better in every curve except the band-curvature family, which drifts per doubling: Hall 4–8%, Nernst 6–8%, `inv_mass` 5–11% (measured with a 25-shell cutoff). This is the known per-direction resolution of degenerate groups in `band_curvature_k` (see the README).

In [ ]:
import os
import sys

# the shared modules live two levels up, in benchmarks/sparse_scaling
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from plots import Benchmark

bench = Benchmark('fe', root=os.path.abspath('runs'), symlog=False)
show = bench.show

## 1. Density of states

The DoS per primitive cell. The 3s and 3p semicore states (down to −89 eV) are below the plotted
range. The PDOS is not compared (spinor copies share a file prefix, see the Pt notebook).

In [ ]:
show(['dos'], ncol=1)

## 2. Band structure at d = 0

Doubling folds the band path, so only the undoubled cell is compared: the dense reference against
`sparse` d = 0, on PAOFLOW's default bcc path. The sparse path holds the
well-projected bands, and NaN above the usable top (the PAO shift minus 0.2 eV). The right panel
is the difference per band within ±3 eV of E_F. `collect.py` reports the RMS over every band as
`dev_bands_vs_ref`, in eV.

Measured with the band path alone (2026-10-26): with the exact H(R) the sparse bands lie within 7
meV of the dense ones within 1 eV of E_F (0.9 meV RMS) and 49 meV anywhere, which is the off-grid
interpolation convention. The largest shell cutoff the QE grid allows (25 shells) would move them
by up to 140 meV near E_F and 2.7 eV elsewhere, which is why Fe runs without one.

In [ ]:
bench.show_bands()

## 3. Boltzmann transport

Constant relaxation time, at 300 K, against the chemical potential. The Hall tensor trace needs
the band curvature, so only `sparse_curvature` has it.

In [ ]:
show(['sigma_xx', 'sigma_zz', 'seebeck_xx', 'kappa_xx', 'pf_xx', 'hall_trace'])

## 4. Hall and Nernst tensors

The Hall components with three different indices, and two Nernst components. Both need the band
curvature, so only `sparse_curvature` has them.

In [ ]:
show(['hall_xyz', 'hall_xzy', 'hall_yzx', 'nernst_xyz', 'nernst_yxz'])

## 5. Band-diagonal and Kubo conductivity

`conductivity` sums v_i v_j over states with adaptive smearing; `linear_response(response='cond')`
is the Kubo formula with a Lorentzian broadening of 0.05 eV, which sums over every pair of
states, so only `sparse_curvature` runs it.

In [ ]:
show(['cond_xx', 'cond_zz', 'lr_cond_xx', 'lr_cond_zz'], ncol=4)

## 6. Anomalous Hall conductivity

σ_xy by the Berry-curvature route (`anomalous_Hall`) and by the Kubo route
(`linear_response(response='ahc')`). σ_yz, which the magnetization along z zeroes, is in the
last section.

In [ ]:
show(['berry_ahc_xy', 'lr_ahc_xy'], ncol=2)

## 7. Magnetic circular dichroism

The frequency-dependent σ_xy of `anomalous_Hall(do_ac=True)`.

In [ ]:
show(['mcd_re_xy', 'mcd_im_xy'], ncol=2)

## 8. Optical response

Kubo-Greenwood ε″ (interband only: the Drude term of a metal is not in the sum). The grey line
is the window top minus 0.5 eV: above it a window run misses the transitions into states above
the window, and ε′ and the curves built from it need transitions at all energies.

In [ ]:
show(['epsi_xx', 'epsi_zz', 'epsr_xx', 'eels_xx', 'refl_xx', 'sigmar_xx'])

## 9. Effective masses

Smeared densities per k-point of the primitive cell, of the inverse DoS mass and of the mean
inverse principal mass (see the Si notebook for why only these symmetric functions compare).

In [ ]:
show(['inv_mass_dos', 'inv_mass_trace'], ncol=2)

## 10. Components that must vanish

With the magnetization along z the bands keep a tetragonal symmetry: xx = yy differs slightly from
zz, but the off-diagonal Boltzmann and conductivity components and the Hall and Nernst components
with a repeated index still vanish, and so does the anomalous Hall σ_yz. The anomalous Hall σ_xy
is not a zero here (it is in `SYSTEM['nonzero']`).

In [ ]:
show(
    [
        'sigma_xy',
        'sigma_xz',
        'sigma_yz',
        'seebeck_xy',
        'kappa_xy',
        'pf_xy',
        'hall_zero',
        'nernst_zero',
        'cond_xy',
        'lr_cond_xy',
        'berry_ahc_yz',
        'lr_ahc_yz',
    ],
    normalized=True,
)

## 11. What is not tested here

- **The spin and orbital Hall conductivities**: Pt tests them.
- **`doping`**: Fe is a metal.
- **PDOS**: spinor copies share a file prefix (see the Pt notebook).